# Listening in on coding sessions with Python

Run the cells in order with **Run → Run All Cells**, or step through them one by one. This workbook uses the bundled **synthetic** 10-call dataset; its commands, timings and token increments are invented. It makes no network requests and does not run a coding agent.

You will inspect repeated calls, elapsed time and token increments using pandas and Matplotlib.

## 1. Load the data

Start Jupyter from the repository root and use the Python environment where `requirements-notebook.txt` was installed. The existing `investigate` function supplies the same validation, sorting and calculations as the command-line demo.

### The raw CSV before pandas

These are the first three data rows exactly as stored, including the header. Each line is one invented tool call; an empty `exit_code` means no result was observed. This is a synthetic CSV fixture, not raw OpenTelemetry.

In [1]:
from pathlib import Path

csv_path = Path('data') / 'events_synthetic.csv'
with csv_path.open(encoding='utf-8') as stream:
    for _ in range(4):  # Header plus three raw rows
        print(next(stream).rstrip('\n'))

session_id,event_id,started_at,tool,command,result,exit_code,tokens,duration_s,provenance
synthetic-demo,e01,2026-01-01T00:00:00+00:00,read,example.py,source-v1,,120,1,synthetic
synthetic-demo,e02,2026-01-01T00:00:10+00:00,edit,example.py,source-v2,,800,4,synthetic
synthetic-demo,e03,2026-01-01T00:00:20+00:00,test,python -m unittest test_fast,5 tests OK,0,400,2,synthetic


In [ ]:
from pathlib import Path
import tempfile
import pandas as pd
from IPython.display import Image, display
from analyze import investigate, plot

raw = pd.read_csv(Path('data') / 'events_synthetic.csv')
df, checks = investigate(raw)
keys = ['session_id', 'tool', 'command']
print('Raw DataFrame: all original columns')
display(raw.head(4))
print(f'Analyzed DataFrame: {len(df)} calls in {df.session_id.nunique()} synthetic session(s)')
display(df[['event_id', 'tool', 'command', 'result', 'exit_code', 'tokens',
            'elapsed_s', 'tool_gap_s', 'cumulative_tokens', 'repeat_index']])

`raw` contains the CSV columns. `investigate` validates and sorts those rows, then adds derived columns such as elapsed time, tool gaps, cumulative token increments and repeat indicators. The `checks` DataFrame contains PR check rows and their check-in gaps.

## 2. Find repeated calls

A repeat is the same session, tool and exact command after its first occurrence. It is a separate event, not a duplicate row. Repetition alone does not prove waste.

In [ ]:
display(df.loc[df.repeat_index.gt(0),
               ['event_id', 'tool', 'command', 'result', 'repeat_index']])
print('Repeated calls:', int(df.repeat_index.gt(0).sum()))

## 3. Inspect PR check-ins and gaps

`check_gap_s` is the time between the starts of successive calls with the same key. The first check has no observed preceding interval. The final check is still a repeat even though its result changes.

In [ ]:
display(checks[['event_id', 'result', 'repeat_index',
                'same_result_as_previous', 'check_gap_s']])
print('Median check-in gap (seconds):', checks.check_gap_s.median())

The underlying pandas operations group by the same key and calculate counts and time differences. `started_at` is already sorted within each session.

In [ ]:
display(pd.DataFrame({
    'event_id': checks.event_id,
    'repeat_from_cumcount': checks.groupby(keys).cumcount(),
    'gap_from_diff': checks.groupby(keys).started_at.diff().dt.total_seconds(),
}))

## 4. Token increments and the chart

These are invented incremental token values. The cumulative series is the sum of increments in each session; it is not a measured cost or the size of a unique context.

In [ ]:
display(df[['event_id', 'tool', 'tokens', 'cumulative_tokens', 'tool_gap_s']])
print('Total increments:', int(df.tokens.sum()))
print('PR check increments:', int(checks.tokens.sum()))

# Use the same plotting function as analyze.py; the temporary image is not saved in the repo.
with tempfile.TemporaryDirectory() as tmp:
    chart = Path(tmp) / 'investigation.png'
    plot(df, checks, chart)
    display(Image(filename=str(chart)))